# LeetCode 191 — Number of 1 Bits bằng Python

**Link đề:** https://leetcode.com/problems/number-of-1-bits/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation (thao tác bit), Divide and Conquer

```python
class Solution:
    def hammingWeight(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu số nguyên được lưu dưới dạng **nhị phân** (binary) và "set bit" (bit bằng 1) là gì.
2. Biết dùng `bin()`, `str.count()` để có lời giải trực giác.
3. Nắm 4 toán tử bit nền tảng: `&`, `>>`, `<<`, `-` kết hợp `&` (mẹo `n & (n - 1)`).
4. Hiểu 4 lời giải: chuỗi → duyệt từng bit → Brian Kernighan → bảng tra byte (follow-up), và `int.bit_count()`.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, tự giải các biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — đếm số bit 1

Cho một số nguyên **dương** `n`. Hãy trả về số lượng bit bằng `1` trong biểu diễn nhị phân của `n`. Con số này còn gọi là **Hamming weight** (trọng số Hamming) hay **popcount**.

| Input `n` | Nhị phân | Output | Giải thích |
|---:|---|:---:|---|
| `11` | `1011` | `3` | có ba chữ số 1 |
| `128` | `10000000` | `1` | chỉ một chữ số 1 |
| `2147483645` | `1111111111111111111111111111101` | `30` | 31 bit, chỉ một bit 0 |

**Constraints:** `1 <= n <= 2^31 - 1`.

**Follow up:** Nếu hàm được gọi rất nhiều lần, ta tối ưu thế nào? (sẽ trả lời ở mục 9)

Câu hỏi đặt ra trước khi code:
- "Bit" là gì? Làm sao "nhìn" thấy bit của một số trong Python?
- Có cách nào đếm mà **không** phải đổi sang chuỗi?
- Có cách nào chỉ tốn số vòng lặp đúng bằng số bit 1, không phải bằng tổng số bit?

## 2. Nền tảng: hệ nhị phân (binary)

Hệ thập phân dùng 10 chữ số `0..9`, mỗi vị trí mạnh gấp 10 lần vị trí bên phải.
Hệ nhị phân chỉ có 2 chữ số `0, 1`, mỗi vị trí mạnh gấp **2** lần vị trí bên phải.

Ví dụ `1011` (nhị phân):

| Vị trí (từ phải, bắt đầu 0) | 3 | 2 | 1 | 0 |
|---|---|---|---|---|
| Bit | 1 | 0 | 1 | 1 |
| Giá trị vị trí | 8 | 4 | 2 | 1 |
| Đóng góp | 8 | 0 | 2 | 1 |

Tổng `8 + 0 + 2 + 1 = 11`. Vậy `11` (thập phân) = `1011` (nhị phân), có **3 bit 1**.

Mỗi chữ số nhị phân gọi là một **bit**. Bit ở vị trí 0 (bên phải nhất) gọi là **bit thấp nhất** (least significant bit, LSB).

In [1]:
# Tự tính lại 11 từ các bit 1, 0, 1, 1
bits = [1, 0, 1, 1]          # đọc từ trái sang phải
value = 0
for bit in bits:
    value = value * 2 + bit  # giống cách đọc số thập phân: value*10 + chữ số
    print("đọc bit", bit, "-> value =", value)
print("Kết quả:", value)

đọc bit 1 -> value = 1
đọc bit 0 -> value = 2
đọc bit 1 -> value = 5
đọc bit 1 -> value = 11
Kết quả: 11


### 2.1. `bin(n)` — xem biểu diễn nhị phân

`bin(n)` trả về **chuỗi** bắt đầu bằng tiền tố `'0b'` (đánh dấu "đây là nhị phân"), sau đó là các bit.

- `bin(11)` → `'0b1011'`
- `bin(128)` → `'0b10000000'`

Để bỏ tiền tố, cắt chuỗi từ vị trí 2: `bin(n)[2:]`. Ngoài ra `format(n, 'b')` cho chuỗi bit không có tiền tố, `format(n, '032b')` đệm đủ 32 ký tự bằng số 0 bên trái.

In [2]:
for n in [11, 128, 5, 1]:
    print(n, "->", bin(n), "| bỏ '0b':", bin(n)[2:], "| format:", format(n, 'b'))
print("Đệm đủ 32 bit:", format(11, '032b'))
print("Kiểu dữ liệu của bin(11):", type(bin(11)))

11 -> 0b1011 | bỏ '0b': 1011 | format: 1011
128 -> 0b10000000 | bỏ '0b': 10000000 | format: 10000000
5 -> 0b101 | bỏ '0b': 101 | format: 101
1 -> 0b1 | bỏ '0b': 1 | format: 1
Đệm đủ 32 bit: 00000000000000000000000000001011
Kiểu dữ liệu của bin(11): <class 'str'>


### 2.2. `str.count(sub)` — đếm số lần xuất hiện

`"banana".count("a")` trả về `3`: số lần chuỗi con `"a"` xuất hiện. Ta sẽ dùng nó để đếm ký tự `'1'` trong chuỗi bit.

Lưu ý: `'0b'` không chứa ký tự `'1'`, nên `bin(n).count('1')` không bị tiền tố làm sai.

In [3]:
print("banana".count("a"))
print("1011".count("1"))
print(bin(11), "có", bin(11).count("1"), "ký tự '1'")
print("'0b'.count('1') =", "0b".count("1"))

3
3
0b1011 có 3 ký tự '1'
'0b'.count('1') = 0


## 3. Cách 1 — Đổi sang chuỗi rồi đếm (trực giác nhất)

Ý tưởng tự nhiên của người mới: "muốn đếm bit 1 thì viết số ra nhị phân rồi đếm chữ số 1".

```python
return bin(n).count('1')
```

Ưu điểm: một dòng, dễ đúng. Nhược điểm: tạo chuỗi mới tốn bộ nhớ `O(số bit)`, và không dạy ta gì về thao tác bit — kỹ năng cần cho rất nhiều bài khác (Reverse Bits, Single Number, Counting Bits...).

In [4]:
def hamming_weight_string(n: int) -> int:
    return bin(n).count('1')

for n in [11, 128, 2147483645]:
    print(n, "->", hamming_weight_string(n))

11 -> 3
128 -> 1
2147483645 -> 30


## 4. Toán tử bit — bộ đồ nghề cho cách không dùng chuỗi

Ta cần 2 thao tác: **đọc bit cuối** và **bỏ bit cuối**.

### 4.1. `n & 1` — đọc bit thấp nhất

`&` là **AND theo từng bit**: bit kết quả bằng 1 chỉ khi **cả hai** bit tương ứng đều là 1.

```text
  1011   (11)
& 0001   (1)
------
  0001   -> 1   (bit cuối của 11 là 1)
```

Vì `1` chỉ có bit cuối bằng 1, `n & 1` "che" hết các bit khác, chỉ giữ lại bit cuối. Kết quả luôn là `0` hoặc `1`. Nó tương đương `n % 2` với số không âm.

In [5]:
for n in [11, 10, 7, 8]:
    print(f"n={n:>2} ({n:04b}) -> n & 1 = {n & 1}, n % 2 = {n % 2}")
print("Ví dụ AND tổng quát: 0b1100 & 0b1010 =", bin(0b1100 & 0b1010))

n=11 (1011) -> n & 1 = 1, n % 2 = 1
n=10 (1010) -> n & 1 = 0, n % 2 = 0
n= 7 (0111) -> n & 1 = 1, n % 2 = 1
n= 8 (1000) -> n & 1 = 0, n % 2 = 0
Ví dụ AND tổng quát: 0b1100 & 0b1010 = 0b1000


### 4.2. `n >> 1` — dịch phải, bỏ bit cuối

`>>` là **dịch phải** (right shift). `n >> 1` đẩy tất cả bit sang phải 1 vị trí, bit cuối "rơi" mất. Với số không âm, `n >> 1` bằng `n // 2`.

```text
1011 (11)  >> 1  ->  101 (5)
101  (5)   >> 1  ->  10  (2)
```

`<<` là dịch trái: `n << 1` bằng `n * 2` (thêm một bit 0 vào cuối). Ta sẽ dùng `1 << i` để tạo số chỉ có bit thứ `i` bằng 1.

In [6]:
n = 11
while n > 0:
    print(f"n={n:>2} = {n:04b} | bit cuối = {n & 1} | n >> 1 = {n >> 1} | n // 2 = {n // 2}")
    n >>= 1          # viết tắt của n = n >> 1
print("Dịch trái: 1 << 0, 1 << 3, 1 << 5 =", 1 << 0, 1 << 3, 1 << 5)

n=11 = 1011 | bit cuối = 1 | n >> 1 = 5 | n // 2 = 5
n= 5 = 0101 | bit cuối = 1 | n >> 1 = 2 | n // 2 = 2
n= 2 = 0010 | bit cuối = 0 | n >> 1 = 1 | n // 2 = 1
n= 1 = 0001 | bit cuối = 1 | n >> 1 = 0 | n // 2 = 0
Dịch trái: 1 << 0, 1 << 3, 1 << 5 = 1 8 32


### 4.3. `>>=`, `+=` — toán tử gán rút gọn

- `n >>= 1` nghĩa là `n = n >> 1`.
- `count += 1` nghĩa là `count = count + 1`.
- `count += n & 1` nghĩa là cộng thêm bit cuối (0 hoặc 1) vào bộ đếm.

Tất cả đều là cú pháp Python bình thường, chỉ viết gọn hơn.

## 5. Cách 2 — Duyệt từng bit (lấy bit cuối, dịch phải)

Ghép hai thao tác ở mục 4:

1. Đọc bit cuối: `n & 1`. Nếu là 1 thì tăng `count`.
2. Bỏ bit cuối: `n >>= 1`.
3. Lặp cho đến khi `n == 0` (không còn bit 1 nào).

Đây là lời giải "đếm thủ công" — tương tự đếm chữ số của số thập phân bằng `% 10` và `// 10`.

In [7]:
def hamming_weight_shift(n: int) -> int:
    count = 0
    while n > 0:
        count += n & 1   # bit cuối là 1 thì cộng 1, là 0 thì cộng 0
        n >>= 1          # bỏ bit cuối
    return count

for n in [11, 128, 2147483645]:
    print(n, "->", hamming_weight_shift(n))

11 -> 3
128 -> 1
2147483645 -> 30


### 5.1. Biến thể "kiểm tra 32 vị trí bằng mặt nạ" (mask)

Thay vì dịch `n`, ta giữ `n` nguyên và dịch một **mặt nạ** `mask = 1 << i` qua 32 vị trí. `n & mask != 0` nghĩa là bit thứ `i` của `n` bằng 1.

Cách này luôn chạy đúng 32 vòng — giống cách viết trong C/Java khi số nguyên có độ rộng cố định 32 bit.

In [8]:
def hamming_weight_mask(n: int) -> int:
    count = 0
    for i in range(32):
        mask = 1 << i
        if n & mask:      # khác 0 -> bit i bằng 1
            count += 1
    return count

print(hamming_weight_mask(11), hamming_weight_mask(128), hamming_weight_mask(2147483645))

3 1 30


## 6. Nút thắt — số vòng lặp phụ thuộc **tổng số bit**, không phải **số bit 1**

Với `n = 128 = 10000000`, cách 2 chạy **8 vòng** nhưng chỉ có **1** bit 1. 7 vòng còn lại chỉ để đi qua các bit 0.

Câu hỏi: có phép nào **nhảy thẳng** đến bit 1 tiếp theo, mỗi vòng xóa đúng một bit 1 không?
Nếu có, số vòng lặp = số bit 1 = chính đáp án.

In [9]:
def count_loops_shift(n: int) -> int:
    loops = 0
    while n > 0:
        n >>= 1
        loops += 1
    return loops

for n in [128, 2**30, 11]:
    print(f"n={n}: số vòng cách 2 = {count_loops_shift(n)}, số bit 1 = {bin(n).count('1')}")

n=128: số vòng cách 2 = 8, số bit 1 = 1
n=1073741824: số vòng cách 2 = 31, số bit 1 = 1
n=11: số vòng cách 2 = 4, số bit 1 = 3


## 7. Insight — `n & (n - 1)` xóa bit 1 thấp nhất

Quan sát phép trừ 1 trong nhị phân. Lấy `n = 12 = 1100`:

```text
n     = 1100
n - 1 = 1011     <- bit 1 thấp nhất (vị trí 2) thành 0, các bit 0 bên phải nó thành 1
n & (n-1):
  1100
& 1011
------
  1000           <- đúng bằng n nhưng mất bit 1 thấp nhất
```

**Vì sao?** Khi trừ 1, ta phải "mượn" từ bit 1 thấp nhất: bit đó thành 0, mọi bit 0 phía bên phải nó thành 1, phần bên trái **giữ nguyên**. AND với `n`:
- phần bên trái: giống nhau → giữ nguyên;
- bit 1 thấp nhất: `1 & 0 = 0` → bị xóa;
- các bit bên phải: `0 & 1 = 0` → vẫn là 0.

Kết luận: **mỗi lần `n = n & (n - 1)` xóa đúng một bit 1**. Kỹ thuật này mang tên **Brian Kernighan**.

In [10]:
for n in [12, 11, 8, 7]:
    print(f"n   = {n:>2} = {n:04b}")
    print(f"n-1 = {n-1:>2} = {n-1:04b}")
    print(f"n&(n-1)  = {n & (n-1):04b}  (xóa bit 1 thấp nhất)")
    print("-" * 28)

n   = 12 = 1100
n-1 = 11 = 1011
n&(n-1)  = 1000  (xóa bit 1 thấp nhất)
----------------------------
n   = 11 = 1011
n-1 = 10 = 1010
n&(n-1)  = 1010  (xóa bit 1 thấp nhất)
----------------------------
n   =  8 = 1000
n-1 =  7 = 0111
n&(n-1)  = 0000  (xóa bit 1 thấp nhất)
----------------------------
n   =  7 = 0111
n-1 =  6 = 0110
n&(n-1)  = 0110  (xóa bit 1 thấp nhất)
----------------------------


### 7.1. Hệ quả nhỏ nhưng hữu ích: kiểm tra lũy thừa của 2

Số là lũy thừa của 2 (`1, 2, 4, 8, ...`) có **đúng một** bit 1. Xóa bit đó thì còn 0. Vậy `n > 0 and n & (n - 1) == 0` ⇔ `n` là lũy thừa của 2. Đây là dấu hiệu cho thấy mẹo này dùng được ở nhiều bài khác (LeetCode 231 Power of Two).

In [11]:
def is_power_of_two(n: int) -> bool:
    return n > 0 and n & (n - 1) == 0

print([x for x in range(1, 70) if is_power_of_two(x)])

[1, 2, 4, 8, 16, 32, 64]


## 8. Cách 3 — Lời giải chính: Brian Kernighan

Ghép insight thành thuật toán, từng đoạn:

**Đoạn 1 — khởi tạo bộ đếm**
```python
count = 0
```

**Đoạn 2 — còn bit 1 nào thì lặp**
```python
while n:          # n != 0 nghĩa là vẫn còn ít nhất một bit 1
```

**Đoạn 3 — xóa một bit 1 và đếm**
```python
    n &= n - 1    # n = n & (n - 1)
    count += 1
```

**Đoạn 4 — trả kết quả**
```python
return count
```

`while n:` dùng **truthiness** (tính đúng/sai) của số: `0` được coi là `False`, mọi số khác 0 là `True`.

In [12]:
print("bool(0) =", bool(0), "| bool(5) =", bool(5))

def hamming_weight_kernighan(n: int) -> int:
    count = 0
    while n:
        n &= n - 1   # xóa bit 1 thấp nhất
        count += 1
    return count

for n in [11, 128, 2147483645]:
    print(n, "->", hamming_weight_kernighan(n))

bool(0) = False | bool(5) = True
11 -> 3
128 -> 1
2147483645 -> 30


## 9. TRACE từng bước

### 9.1. Bảng trace bằng tay cho `n = 11`

| Vòng | `n` trước | `n - 1` | `n & (n-1)` | Bit bị xóa | `count` sau |
|---:|---|---|---|---|---:|
| 1 | `1011` (11) | `1010` (10) | `1010` (10) | vị trí 0 | 1 |
| 2 | `1010` (10) | `1001` (9) | `1000` (8) | vị trí 1 | 2 |
| 3 | `1000` (8) | `0111` (7) | `0000` (0) | vị trí 3 | 3 |
| — | `0000` | | | dừng vì `n == 0` | **3** |

Ý nghĩa các cột: `n trước` là trạng thái đầu vòng; cột `n & (n-1)` là giá trị mới của `n`; vòng lặp dừng khi `n` về 0.

In [13]:
def trace_kernighan(n: int) -> int:
    print(f"Bắt đầu: n = {n} = {n:b}")
    count = 0
    step = 1
    while n:
        new_n = n & (n - 1)
        removed = n ^ new_n          # ^ là XOR: chỉ còn lại bit vừa bị xóa
        count += 1
        print(f"Vòng {step}: n={n:>4b}  n-1={n-1:>4b}  n&(n-1)={new_n:>4b}  "
              f"xóa bit giá trị {removed}  count={count}")
        n = new_n
        step += 1
    print("Dừng vì n == 0, đáp án =", count)
    return count

trace_kernighan(11)

Bắt đầu: n = 11 = 1011
Vòng 1: n=1011  n-1=1010  n&(n-1)=1010  xóa bit giá trị 1  count=1
Vòng 2: n=1010  n-1=1001  n&(n-1)=1000  xóa bit giá trị 2  count=2
Vòng 3: n=1000  n-1= 111  n&(n-1)=   0  xóa bit giá trị 8  count=3
Dừng vì n == 0, đáp án = 3


3

### 9.2. Trace so sánh: `n = 128` với cách 2 và cách 3

Cách 2 phải đi qua 8 vị trí; cách 3 chỉ 1 vòng. Hàm trace dưới in quyết định từng bước của cách 2 để thấy rõ các vòng "phí".

In [14]:
def trace_shift(n: int) -> int:
    count = 0
    step = 1
    while n > 0:
        bit = n & 1
        count += bit
        print(f"Vòng {step}: n={n:>8b} bit cuối={bit} -> count={count}")
        n >>= 1
        step += 1
    return count

trace_shift(128)
print()
trace_kernighan(128)

Vòng 1: n=10000000 bit cuối=0 -> count=0
Vòng 2: n= 1000000 bit cuối=0 -> count=0
Vòng 3: n=  100000 bit cuối=0 -> count=0
Vòng 4: n=   10000 bit cuối=0 -> count=0
Vòng 5: n=    1000 bit cuối=0 -> count=0
Vòng 6: n=     100 bit cuối=0 -> count=0
Vòng 7: n=      10 bit cuối=0 -> count=0
Vòng 8: n=       1 bit cuối=1 -> count=1

Bắt đầu: n = 128 = 10000000
Vòng 1: n=10000000  n-1=1111111  n&(n-1)=   0  xóa bit giá trị 128  count=1
Dừng vì n == 0, đáp án = 1


1

### 9.3. Trace ví dụ lớn — `n = 2147483645`

Không in từng vòng (30 vòng), chỉ đếm số vòng để xác nhận: số vòng của Kernighan đúng bằng đáp án 30, còn cách dịch bit chạy 31 vòng (tổng số bit).

In [15]:
n = 2147483645
loops_kernighan = 0
x = n
while x:
    x &= x - 1
    loops_kernighan += 1
print("Số bit của n:", n.bit_length())
print("Vòng cách dịch bit:", count_loops_shift(n))
print("Vòng Kernighan:", loops_kernighan)

Số bit của n: 31
Vòng cách dịch bit: 31
Vòng Kernighan: 30


## 10. Cách 4 — Follow-up: gọi rất nhiều lần → bảng tra (lookup table)

Nếu hàm bị gọi hàng triệu lần, ta có thể **tính trước** số bit 1 cho mọi số 8 bit (`0..255`, gọi là 1 **byte**), lưu vào list `TABLE`. Một số 32 bit gồm 4 byte; đếm mỗi byte bằng một lần tra bảng rồi cộng lại.

Công thức tạo bảng rất đẹp: `TABLE[i] = TABLE[i >> 1] + (i & 1)` — số bit 1 của `i` bằng số bit 1 của `i` bỏ bit cuối, cộng bit cuối.

- `n & 0xFF` lấy 8 bit thấp nhất (`0xFF` = `255` = `11111111`, viết dạng hexa).
- `n >> 8` bỏ 8 bit thấp nhất để chuyển sang byte tiếp theo.

In [16]:
TABLE = [0] * 256
for i in range(1, 256):
    TABLE[i] = TABLE[i >> 1] + (i & 1)
print("TABLE[0..8] =", TABLE[:9], "| TABLE[255] =", TABLE[255])
print("0xFF =", 0xFF)

def hamming_weight_table(n: int) -> int:
    count = 0
    while n:
        count += TABLE[n & 0xFF]   # đếm 8 bit thấp bằng một lần tra
        n >>= 8                    # sang byte tiếp theo
    return count

for n in [11, 128, 2147483645]:
    print(n, "->", hamming_weight_table(n))

TABLE[0..8] = [0, 1, 1, 2, 1, 2, 2, 3, 1] | TABLE[255] = 8
0xFF = 255
11 -> 3
128 -> 1
2147483645 -> 30


### 10.1. Cách 5 — Hàm có sẵn `int.bit_count()` (Python 3.10+)

Python có sẵn phương thức đếm popcount: `n.bit_count()`. Rất nhanh (viết bằng C). Khi phỏng vấn, nên nói biết hàm này **và** trình bày được cách Kernighan.

In [17]:
for n in [11, 128, 2147483645]:
    print(n, "->", n.bit_count())

11 -> 3
128 -> 1
2147483645 -> 30


## 11. Code nộp LeetCode

Bản chính (Brian Kernighan). Signature giữ đúng `hammingWeight(self, n: int) -> int`.

In [18]:
class Solution:
    def hammingWeight(self, n: int) -> int:
        count = 0
        while n:
            n &= n - 1
            count += 1
        return count

sol = Solution()
print(sol.hammingWeight(11), sol.hammingWeight(128), sol.hammingWeight(2147483645))

3 1 30


Các bản thay thế cũng được chấp nhận (chọn một khi nộp):

```python
class Solution:
    def hammingWeight(self, n: int) -> int:
        return bin(n).count('1')          # Cách 1
```

```python
class Solution:
    def hammingWeight(self, n: int) -> int:
        count = 0
        while n:
            count += n & 1                # Cách 2
            n >>= 1
        return count
```

## 12. Vì sao thuật toán Kernighan đúng? (invariant)

Gọi `n0` là giá trị ban đầu. **Bất biến (invariant)** ở đầu mỗi vòng lặp:

> `count + popcount(n) == popcount(n0)`

- **Ban đầu:** `count = 0`, `n = n0` → đúng.
- **Mỗi vòng:** `n & (n - 1)` xóa đúng một bit 1 (đã chứng minh ở mục 7) nên `popcount(n)` giảm 1, đồng thời `count` tăng 1 → tổng không đổi.
- **Kết thúc:** vòng lặp dừng khi `n == 0`, tức `popcount(n) = 0` → `count = popcount(n0)`.
- **Dừng chắc chắn:** mỗi vòng `n` mất một bit 1, mà `n` chỉ có hữu hạn bit 1 → vòng lặp hữu hạn.

Với cách 2, invariant tương tự: `count + popcount(n) == popcount(n0)`, vì dịch phải chỉ bỏ bit cuối mà ta vừa cộng vào `count`.

## 13. Phân tích Time / Space và so sánh

Gọi `b` = số bit của `n` (ở bài này `b <= 31`), `k` = số bit 1 (`k <= b`).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Chuỗi | `bin(n).count('1')` | `O(b)` | `O(b)` | ngắn nhất, tạo chuỗi |
| 2. Dịch bit | `n & 1`, `n >>= 1` | `O(b)` | `O(1)` | nền tảng, dễ trace |
| 2'. Mặt nạ 32 vị trí | `n & (1 << i)` | `O(32)` | `O(1)` | luôn 32 vòng |
| 3. Kernighan | `n &= n - 1` | `O(k)` | `O(1)` | số vòng = đáp án |
| 4. Bảng byte | tra `TABLE[n & 0xFF]` | `O(b/8)` | `O(256)` | tốt khi gọi rất nhiều lần |
| 5. `n.bit_count()` | hàm có sẵn | `O(b)` (nhanh, trong C) | `O(1)` | Python 3.10+ |

Vì `b <= 31` là hằng số, mọi cách đều là `O(1)` theo nghĩa chặt. Nhưng khi nói về số nguyên `b` bit tổng quát, bảng trên mới cho thấy khác biệt.

## 14. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

Nhóm test:
1. Ba ví dụ của đề.
2. Biên: `1` (nhỏ nhất), `2^31 - 1` (lớn nhất, 31 bit 1).
3. Lũy thừa của 2 (chỉ 1 bit 1) và `2^k - 1` (toàn bit 1).
4. Đối chiếu tất cả các cách trên mọi `n` từ 1 đến 5000 và trên số ngẫu nhiên có seed.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("ví dụ 1 (11)", sol.hammingWeight(11), 3)
check("ví dụ 2 (128)", sol.hammingWeight(128), 1)
check("ví dụ 3 (2147483645)", sol.hammingWeight(2147483645), 30)
check("nhỏ nhất (1)", sol.hammingWeight(1), 1)
check("lớn nhất (2^31-1)", sol.hammingWeight(2**31 - 1), 31)
check("lũy thừa 2 (2^30)", sol.hammingWeight(2**30), 1)
check("toàn bit 1 (255)", sol.hammingWeight(255), 8)
check("xen kẽ 0b1010101", sol.hammingWeight(0b1010101), 4)

PASS ví dụ 1 (11): 3
PASS ví dụ 2 (128): 1
PASS ví dụ 3 (2147483645): 30
PASS nhỏ nhất (1): 1
PASS lớn nhất (2^31-1): 31
PASS lũy thừa 2 (2^30): 1
PASS toàn bit 1 (255): 8
PASS xen kẽ 0b1010101: 4


In [20]:
import random

methods = [hamming_weight_string, hamming_weight_shift, hamming_weight_mask,
           hamming_weight_kernighan, hamming_weight_table, lambda x: x.bit_count()]

for n in range(1, 5001):
    results = {m(n) for m in methods}
    assert len(results) == 1, (n, results)

rng = random.Random(191)
for _ in range(2000):
    n = rng.randint(1, 2**31 - 1)
    results = {m(n) for m in methods}
    assert len(results) == 1, (n, results)
print("Tất cả 6 cách cho cùng kết quả trên 7000 input!")

Tất cả 6 cách cho cùng kết quả trên 7000 input!


## 15. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `n / 2` thay vì `n >> 1` hoặc `n // 2`

```python
n = n / 2      # SAI: tạo số thực 5.5, 2.75, ...
```
`/` là phép chia thực, `n` thành `float`, `n & 1` sẽ báo lỗi `TypeError`. Sửa: `n >>= 1` hoặc `n //= 2`.

In [21]:
n = 11
print("11 / 2 =", n / 2, "| 11 // 2 =", n // 2, "| 11 >> 1 =", n >> 1)
try:
    (n / 2) & 1
except TypeError as e:
    print("Lỗi khi dùng & với float:", e)

11 / 2 = 5.5 | 11 // 2 = 5 | 11 >> 1 = 5
Lỗi khi dùng & với float: unsupported operand type(s) for &: 'float' and 'int'


### Lỗi 2 — Mang thứ tự ưu tiên toán tử của C/Java sang (và ngược lại)

Trong **C/Java/JavaScript**, `==` được tính **trước** `&`, nên `n & (n - 1) == 0` bị hiểu thành `n & ((n - 1) == 0)` — sai.  
Trong **Python** thì ngược lại: thứ tự ưu tiên là `-` → `&` → `==`, nên `n & (n - 1) == 0` được hiểu đúng là `(n & (n - 1)) == 0`.

Vì người học thường viết nhiều ngôn ngữ, **luôn đặt ngoặc rõ ràng**: `(n & (n - 1)) == 0`. Code dễ đọc và không phụ thuộc trí nhớ về độ ưu tiên.

In [22]:
for n in [6, 8, 1, 3, 16]:
    no_paren = n & (n - 1) == 0          # Python hiểu là (n & (n-1)) == 0
    with_paren = (n & (n - 1)) == 0
    c_style = n & ((n - 1) == 0)         # cách C/Java sẽ hiểu dòng không ngoặc
    print(f"n={n:>2} | Python không ngoặc={no_paren!s:<5} | có ngoặc={with_paren!s:<5} | kiểu C hiểu={c_style}")

n= 6 | Python không ngoặc=False | có ngoặc=False | kiểu C hiểu=0
n= 8 | Python không ngoặc=True  | có ngoặc=True  | kiểu C hiểu=0
n= 1 | Python không ngoặc=True  | có ngoặc=True  | kiểu C hiểu=1
n= 3 | Python không ngoặc=False | có ngoặc=False | kiểu C hiểu=0
n=16 | Python không ngoặc=True  | có ngoặc=True  | kiểu C hiểu=0


Quan sát: hai cột Python luôn giống nhau; cột "kiểu C hiểu" cho kết quả vô nghĩa (`0`/`1`). Bài học: biết độ ưu tiên, nhưng vẫn đặt ngoặc.

### Lỗi 3 — Đếm cả tiền tố `'0b'` hoặc đếm sai ký tự

```python
len(bin(n))         # SAI: đếm tổng số ký tự, gồm cả '0b' và các bit 0
bin(n).count(1)     # SAI: count cần chuỗi '1', không phải số 1 -> TypeError
```
Sửa: `bin(n).count('1')`.

### Lỗi 4 — Vòng lặp vô hạn khi quên cập nhật `n`

```python
while n:
    count += n & 1      # quên n >>= 1 -> n không bao giờ về 0
```
Mỗi vòng lặp `while` phải có một dòng làm **tiến** điều kiện dừng.

### Lỗi 5 — Mang thói quen C/Java với số âm vào Python

Trong Python, số nguyên không giới hạn độ rộng (không có "32 bit" cố định). `-1 >> 1` vẫn là `-1`; với số âm, cách 2 (`while n > 0`) trả `0` ngay (sai), còn Kernighan (`while n:`) lặp mãi vì `n` không bao giờ về 0. Bài này `n` dương nên an toàn; nếu đề cho số âm 32 bit, phải chuẩn hóa `n &= 0xFFFFFFFF` trước.

In [23]:
print("-1 >> 1 =", -1 >> 1, "(không về 0!)")
neg = -3
print("-3 chuẩn hóa 32 bit:", format(neg & 0xFFFFFFFF, '032b'), "->", hamming_weight_kernighan(neg & 0xFFFFFFFF), "bit 1")

-1 >> 1 = -1 (không về 0!)
-3 chuẩn hóa 32 bit: 11111111111111111111111111111101 -> 31 bit 1


## 16. Mẹo nhận dạng pattern

- Đề nói "binary representation", "set bits", "bit 1", "Hamming" → nghĩ ngay tới `& 1`, `>>`, `n & (n - 1)`.
- Cần **đi qua từng bit**: `while n: ... n >>= 1` (giống `% 10`, `// 10` với chữ số thập phân).
- Cần **đi qua từng bit 1**: `while n: n &= n - 1`.
- Hàm bị **gọi nhiều lần** với miền giá trị nhỏ → **tiền tính bảng** (precompute / lookup table).
- Câu hỏi "có phải lũy thừa của 2?" → `n > 0 and (n & (n - 1)) == 0`.
- Bài liên quan: 190 Reverse Bits, 231 Power of Two, 338 Counting Bits, 461 Hamming Distance, 136 Single Number (XOR).

## 17. Biến thể

### 17.1. Hamming Distance (LeetCode 461)
Khoảng cách Hamming giữa `x` và `y` = số vị trí bit khác nhau. `x ^ y` (XOR) cho bit 1 đúng ở các vị trí khác nhau → đếm bit 1 của `x ^ y`.

### 17.2. Counting Bits (LeetCode 338)
Trả về list `ans` với `ans[i]` = số bit 1 của `i` cho mọi `0 <= i <= n`. Dùng đúng công thức tạo bảng: `ans[i] = ans[i >> 1] + (i & 1)` → `O(n)`.

In [24]:
def hamming_distance(x: int, y: int) -> int:
    return hamming_weight_kernighan(x ^ y)

print("XOR 1 ^ 4 =", bin(1 ^ 4), "-> distance =", hamming_distance(1, 4))   # 001 vs 100 -> 2

def counting_bits(n: int) -> list[int]:
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans

print("counting_bits(5) =", counting_bits(5))   # [0,1,1,2,1,2]

XOR 1 ^ 4 = 0b101 -> distance = 2
counting_bits(5) = [0, 1, 1, 2, 1, 2]


## 18. Bài tập tự luyện (4–6 bài, tăng dần)

### Bài 1 — Dự đoán output (không chạy)
Viết ra giấy kết quả của: `bin(13)`, `13 & 1`, `13 >> 2`, `13 & 12`, `(13).bit_count()`.

### Bài 2 — Trace tay
Lập bảng trace Kernighan cho `n = 22`: mỗi vòng ghi `n`, `n - 1`, `n & (n - 1)`, `count`.

### Bài 3 — TODO: tự viết cách dịch bit

In [25]:
def exercise_shift(n: int) -> int:
    # TODO 1: khởi tạo count
    # TODO 2: lặp khi n > 0: cộng bit cuối, rồi dịch phải
    # TODO 3: trả về count
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_shift(11) == 3
# assert exercise_shift(128) == 1

### Bài 4 — TODO: tự viết Brian Kernighan

In [26]:
def exercise_kernighan(n: int) -> int:
    # TODO: mỗi vòng xóa bit 1 thấp nhất và tăng count
    pass

# assert exercise_kernighan(2147483645) == 30
# assert exercise_kernighan(1) == 1

### Bài 5 — Biến thể: Hamming Distance
Viết `exercise_hamming_distance(x, y)` trả số vị trí bit khác nhau giữa `x` và `y`.

In [27]:
def exercise_hamming_distance(x: int, y: int) -> int:
    # TODO: dùng phép XOR rồi đếm bit 1
    pass

# assert exercise_hamming_distance(1, 4) == 2
# assert exercise_hamming_distance(3, 1) == 1

### Bài 6 — Mini challenge: số có cùng số bit 1
Cho list số dương `nums`, trả về dict gom nhóm: key = số bit 1, value = list các số (giữ thứ tự xuất hiện).
Ví dụ `group_by_popcount([3, 5, 8, 7, 1])` → `{2: [3, 5], 1: [8, 1], 3: [7]}`.

In [28]:
def exercise_group_by_popcount(nums: list[int]) -> dict[int, list[int]]:
    # TODO: với mỗi số, tính popcount, thêm vào đúng nhóm
    pass

# assert exercise_group_by_popcount([3, 5, 8, 7, 1]) == {2: [3, 5], 1: [8, 1], 3: [7]}

## Gợi ý 3 mức (chỉ mở khi bí)

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cú pháp | Mức 3 — gần code |
|---|---|---|---|
| 1 | Đổi 13 ra nhị phân trước: `1101` | `&` giữ bit chung, `>>` bỏ bit cuối | `13 & 12` = `1101 & 1100` |
| 2 | Mỗi vòng xóa 1 bit 1 | `22 = 10110` | 3 vòng: 22→20→16→0 |
| 3 | Giống đếm chữ số thập phân | `n & 1`, `n >>= 1` | `while n > 0: count += n & 1; n >>= 1` |
| 4 | Số vòng = số bit 1 | `n &= n - 1` | `while n: n &= n - 1; count += 1` |
| 5 | Bit khác nhau → XOR cho 1 | `x ^ y` | `return exercise_kernighan(x ^ y)` hoặc `.bit_count()` |
| 6 | Dict từ key → list | `d.setdefault(k, []).append(x)` | `k = x.bit_count()` |

## Đáp án tham khảo (đặt cuối để không lộ)

**Bài 1:** `bin(13)` = `'0b1101'`; `13 & 1` = `1`; `13 >> 2` = `3` (`11`); `13 & 12` = `12` (`1100`, chính là xóa bit 1 thấp nhất vì 12 = 13 - 1); `(13).bit_count()` = `3`.

**Bài 2:**

| Vòng | n | n-1 | n&(n-1) | count |
|---:|---|---|---|---:|
| 1 | 10110 (22) | 10101 (21) | 10100 (20) | 1 |
| 2 | 10100 (20) | 10011 (19) | 10000 (16) | 2 |
| 3 | 10000 (16) | 01111 (15) | 00000 (0) | 3 |

In [29]:
def answer_shift(n: int) -> int:
    count = 0
    while n > 0:
        count += n & 1
        n >>= 1
    return count

def answer_kernighan(n: int) -> int:
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count

def answer_hamming_distance(x: int, y: int) -> int:
    return answer_kernighan(x ^ y)

def answer_group_by_popcount(nums: list[int]) -> dict[int, list[int]]:
    groups: dict[int, list[int]] = {}
    for x in nums:
        groups.setdefault(answer_kernighan(x), []).append(x)
    return groups

# Kiểm tra đáp án bài 1, 2
assert bin(13) == '0b1101' and 13 & 1 == 1 and 13 >> 2 == 3 and 13 & 12 == 12 and (13).bit_count() == 3
assert trace_kernighan(22) == 3
# Bài 3–6
assert answer_shift(11) == 3 and answer_shift(128) == 1
assert answer_kernighan(2147483645) == 30 and answer_kernighan(1) == 1
assert answer_hamming_distance(1, 4) == 2 and answer_hamming_distance(3, 1) == 1
assert answer_group_by_popcount([3, 5, 8, 7, 1]) == {2: [3, 5], 1: [8, 1], 3: [7]}
print("Đáp án tham khảo PASS toàn bộ!")

Bắt đầu: n = 22 = 10110
Vòng 1: n=10110  n-1=10101  n&(n-1)=10100  xóa bit giá trị 2  count=1
Vòng 2: n=10100  n-1=10011  n&(n-1)=10000  xóa bit giá trị 4  count=2
Vòng 3: n=10000  n-1=1111  n&(n-1)=   0  xóa bit giá trị 16  count=3
Dừng vì n == 0, đáp án = 3
Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
# Đi qua TỪNG BIT                 # Đi qua TỪNG BIT 1 (Kernighan)
while n:                          while n:
    bit = n & 1                       n &= n - 1     # xóa bit 1 thấp nhất
    ...                               count += 1
    n >>= 1
```

Công thức cần thuộc:
- `n & 1` → bit cuối; `n >> 1` → bỏ bit cuối; `1 << i` → mặt nạ bit `i`.
- `n & (n - 1)` → xóa bit 1 thấp nhất; `(n & (n - 1)) == 0` (với `n > 0`) → lũy thừa của 2.
- `x ^ y` → bit 1 ở vị trí khác nhau.

**Checklist trước khi nộp:**
- [ ] Tên method đúng `hammingWeight`, trả `int`.
- [ ] Dùng `>>` hoặc `//`, **không** dùng `/`.
- [ ] Mọi vòng `while` đều làm `n` tiến về 0.
- [ ] Đặt ngoặc khi trộn `&` với `==`, `-`.
- [ ] Đã thử `n = 1`, `n = 2^31 - 1`, lũy thừa của 2.